In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install  rasterio

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.7 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostClassifier


LABEL_COL = "LABEL"
MODEL_DIR = "models/catboost"
os.makedirs(MODEL_DIR, exist_ok=True)
EPS = 1e-6
BATCH_SIZE = 200_000
BAND_MAP = {
    'B01': 0, 'B02': 1, 'B03': 2, 'B04': 3,
    'B05': 4, 'B06': 5, 'B07': 6, 'B08': 7,
    'B8A': 8, 'B09': 9, 'B10': 10, 'B11': 11, 'B12': 12,
}

# ========= BAND MAP =========
FEATURE_NAMES = [
    'B01','B02','B03','B04','B05','B06','B07','B08','B8A','B09','B11','B12',
    'NDVI','NDSI','NDWI','NDMI',
    'B11_B08_Ratio','B02_B04_Ratio',
    'B8A_B11_Ratio','B01_B11_Ratio',
    'B05_B04_Ratio','B12_B03_Ratio'
]


In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from catboost import CatBoostClassifier
import glob
import joblib

DATA_DIR = "/content/drive/MyDrive/GEE S2_Data/Traning/"
FILE_PATTERN = os.path.join(DATA_DIR, "S_*_TRAINING_data.csv")
LABEL_COL = "LABEL"
MODEL_DIR = "models/catboost"
os.makedirs(MODEL_DIR, exist_ok=True)
EPS = 1e-6
BATCH_SIZE = 1000_000
GROUP_COL = "Tile_ID"
FEATURE_NAMES = [
    'B01','B02','B03','B04','B05','B06','B07','B08','B8A','B09','B11','B12',
    'NDVI','NDSI','NDWI','NDMI',
    'B11_B08_Ratio','B02_B04_Ratio',
    'B8A_B11_Ratio','B01_B11_Ratio',
    'B05_B04_Ratio','B12_B03_Ratio'
]

COLS_TO_LOAD = FEATURE_NAMES + [LABEL_COL, GROUP_COL]

print("Bắt đầu nạp dữ liệu từ các file...")
all_files = glob.glob(FILE_PATTERN)
print(f"Tìm thấy {len(all_files)} file CSV.")

chunk_list = []
total_rows = 0

for file_path in all_files:
    print(f"  -> Nạp file: {os.path.basename(file_path)}")
    for chunk in pd.read_csv(file_path, chunksize=BATCH_SIZE, usecols=COLS_TO_LOAD):
        chunk_list.append(chunk)
        total_rows += len(chunk)
df = pd.concat(chunk_list, ignore_index=True)
del chunk_list

print("CSV shape sau khi kết hợp và lọc:", df.shape)
groups = df[GROUP_COL].values
X = df[FEATURE_NAMES].values.astype(np.float32)
y = df[LABEL_COL].values.astype(np.int64)
del df

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.pkl"))
print(f"Đã lưu Scaler tại: {os.path.join(MODEL_DIR, 'scaler.pkl')}")

cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

oof_probs = np.zeros(len(y), dtype=np.float32)
oof_true = y.copy()



In [ ]:
for fold, (tr_idx, val_idx) in enumerate(cv.split(X_scaled, y, groups)):
    print(f"\n========== FOLD {fold} ==========")

    X_tr, X_val = X_scaled[tr_idx], X_scaled[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = CatBoostClassifier(
        iterations=150,
        depth=8,
        learning_rate=0.1,
        loss_function="Logloss",
        eval_metric="F1",
        random_seed=42,
        verbose=50,
        early_stopping_rounds=20
    )

    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        use_best_model=True
    )
    oof_probs[val_idx] = model.predict_proba(X_val)[:, 1]
    model_path = os.path.join(MODEL_DIR, f"catboost_fold{fold}.cbm")
    model.save_model(model_path)
    print(f"Đã lưu mô hình FOLD {fold} tại: {model_path}")


========== FOLD 0 ==========
0:	learn: 0.8589804	test: 0.8771067	best: 0.8771067 (0)	total: 1.74s	remaining: 5m 45s
50:	learn: 0.8823305	test: 0.8975694	best: 0.8977024 (48)	total: 1m 33s	remaining: 4m 32s
Stopped by overfitting detector  (20 iterations wait)

bestTest = 0.8977300283
bestIteration = 55

Shrink model to first 56 iterations.
Đã lưu mô hình FOLD 0 tại: models/catboost/catboost_fold0.cbm

========== FOLD 1 ==========
0:	learn: 0.8687538	test: 0.8634889	best: 0.8634889 (0)	total: 1.61s	remaining: 5m 19s
50:	learn: 0.8871052	test: 0.8708941	best: 0.8709668 (49)	total: 1m 27s	remaining: 4m 14s
100:	learn: 0.8915545	test: 0.8743305	best: 0.8743305 (100)	total: 2m 49s	remaining: 2m 46s
150:	learn: 0.8937337	test: 0.8763056	best: 0.8763056 (150)	total: 4m 12s	remaining: 1m 21s
199:	learn: 0.8951604	test: 0.8768517	best: 0.8769385 (196)	total: 5m 40s	remaining: 0us

bestTest = 0.876938512
bestIteration = 196

Shrink model to first 197 iterations.
Đã lưu mô hình FOLD 1 tại: model

In [ ]:
!pip install lightgbm

In [ ]:
import lightgbm as lgb
for fold, (tr_idx, val_idx) in enumerate(cv.split(X_scaled, y, groups)):
    print(f"\n========== FOLD {fold} ==========")

    X_tr, X_val = X_scaled[tr_idx], X_scaled[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]


    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)


    params = {
      'objective': 'binary',
      'metric': 'binary_logloss',
      'boosting_type': 'gbdt',
      'learning_rate': 0.1,
      'num_leaves': 50,
      'max_depth': 8,
      'subsample': 0.8,
      'colsample_bytree': 0.8,
      'seed': 42,
      'verbose': -1
      }

    # Huấn luyện
    model = lgb.train(
        params,
        train_data,
        num_boost_round=150,
        valid_sets=[val_data],
    )

    # Dự đoán
    oof_probs[val_idx] = model.predict(X_val, num_iteration=model.best_iteration)

    # Lưu mô hình
    model_path = os.path.join(MODEL_DIR, f"lgb_fold{fold}.txt")
    model.save_model(model_path)
    print(f"Đã lưu mô hình FOLD {fold} tại: {model_path}")


========== FOLD 0 ==========
Đã lưu mô hình FOLD 0 tại: models/catboost/lgb_fold0.txt

========== FOLD 1 ==========
Đã lưu mô hình FOLD 1 tại: models/catboost/lgb_fold1.txt

========== FOLD 2 ==========
Đã lưu mô hình FOLD 2 tại: models/catboost/lgb_fold2.txt


In [ ]:
!pip install xgboost

In [ ]:
import xgboost as xgb
for fold, (tr_idx, val_idx) in enumerate(cv.split(X_scaled, y, groups)):
    print(f"\n========== FOLD {fold} ==========")

    X_tr, X_val = X_scaled[tr_idx], X_scaled[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)

    # Tham số XGBoost (CPU)
    params = {
      'objective': 'binary:logistic',
      'eval_metric': 'logloss',
      'max_depth': 8,
      'learning_rate': 0.1,
      'subsample': 0.8,
      'colsample_bytree': 0.8,
      'seed': 42,
      'verbosity': 1,
      'tree_method': 'hist',
      'nthread': -1
    }

    evals = [(dtrain, 'train'), (dval, 'eval')]

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=150,
        evals=evals,
        early_stopping_rounds=20,
        verbose_eval=50
    )

    oof_probs[val_idx] = model.predict(dval, iteration_range=(0, model.best_iteration))

    model_path = os.path.join(MODEL_DIR, f"xgb_fold{fold}.json")
    model.save_model(model_path)
    print(f"Đã lưu mô hình FOLD {fold} tại: {model_path}")



========== FOLD 0 ==========
[0]	train-logloss:0.63757	eval-logloss:0.64813
[50]	train-logloss:0.31671	eval-logloss:0.29216
[100]	train-logloss:0.29612	eval-logloss:0.27494
[150]	train-logloss:0.29039	eval-logloss:0.27343
[181]	train-logloss:0.28760	eval-logloss:0.27343
Đã lưu mô hình FOLD 0 tại: models/catboost/xgb_fold0.json

========== FOLD 1 ==========
[0]	train-logloss:0.64658	eval-logloss:0.63256
[50]	train-logloss:0.30267	eval-logloss:0.33997
[98]	train-logloss:0.28193	eval-logloss:0.33434
Đã lưu mô hình FOLD 1 tại: models/catboost/xgb_fold1.json

========== FOLD 2 ==========
[0]	train-logloss:0.63625	eval-logloss:0.64845
[50]	train-logloss:0.29196	eval-logloss:0.36476
[100]	train-logloss:0.26952	eval-logloss:0.35404
[118]	train-logloss:0.26719	eval-logloss:0.35410
Đã lưu mô hình FOLD 2 tại: models/catboost/xgb_fold2.json


In [ ]:
from sklearn.utils import resample
import math

MAX_SAMPLES = 500_000

if len(y) > MAX_SAMPLES:
    X_small, y_small, groups_small = resample(
        X_scaled, y, groups,
        n_samples=MAX_SAMPLES,
        random_state=42,
        stratify=y
    )
else:
    X_small, y_small, groups_small = X_scaled, y, groups

oof_probs = np.zeros(len(y_small))

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_small, y_small, groups_small)):
    print(f"\n========== FOLD {fold} ==========")

    X_tr, X_val = X_small[tr_idx], X_small[val_idx]
    y_tr, y_val = y_small[tr_idx], y_small[val_idx]

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=42,
        max_features= math.sqrt(X_tr.shape[1]),
        n_jobs=-1
    )

    model.fit(X_tr, y_tr)
    oof_probs[val_idx] = model.predict_proba(X_val)[:, 1]

    # Lưu model
    model_path = os.path.join(MODEL_DIR, f"rf_fold{fold}.joblib")
    import joblib
    joblib.dump(model, model_path)
    print(f"Đã lưu mô hình RF FOLD {fold} tại: {model_path}")



========== FOLD 0 ==========
Đã lưu mô hình RF FOLD 0 tại: models/catboost/rf_fold0.joblib

========== FOLD 1 ==========
Đã lưu mô hình RF FOLD 1 tại: models/catboost/rf_fold1.joblib

========== FOLD 2 ==========
Đã lưu mô hình RF FOLD 2 tại: models/catboost/rf_fold2.joblib


In [ ]:
import os
import joblib
import numpy as np
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

MAX_SAMPLES = 1_000_000

if len(y) > MAX_SAMPLES:
    X_small, y_small, groups_small = resample(
        X_scaled, y, groups,
        n_samples=MAX_SAMPLES,
        random_state=42,
        stratify=y
    )
else:
    X_small, y_small, groups_small = X_scaled, y, groups

oof_scores = np.zeros(len(y_small))

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_small, y_small, groups_small)):
    print(f"\n========== FOLD {fold} ==========")

    X_tr, X_val = X_small[tr_idx], X_small[val_idx]
    y_tr, y_val = y_small[tr_idx], y_small[val_idx]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", LinearSVC(
            C=1.438,
            tol=0.00251,
            loss="squared_hinge",
            random_state=42,
            max_iter=5000
        ))
    ])

    model.fit(X_tr, y_tr)
    oof_scores[val_idx] = model.decision_function(X_val)
    model_path = os.path.join(MODEL_DIR, f"linear_svm_fold{fold}.joblib")
    joblib.dump(model, model_path)
    print(f"Đã lưu mô hình Linear SVM FOLD {fold} tại: {model_path}")



========== FOLD 0 ==========
Đã lưu mô hình SVM FOLD 0 tại: models/catboost/svm_fold0.joblib

========== FOLD 1 ==========
Đã lưu mô hình SVM FOLD 1 tại: models/catboost/svm_fold1.joblib

========== FOLD 2 ==========
Đã lưu mô hình SVM FOLD 2 tại: models/catboost/svm_fold2.joblib


In [ ]:
from sklearn.metrics import f1_score
def tune_threshold(oof_action, y_action, step=0.01):
    best_thr = 0.5
    best_f1 = -1.0

    for thr in np.arange(0.0, 1.0 + 1e-9, step):
        y_pred = (oof_action >= thr).astype(int)
        f1 = f1_score(y_action, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr

    return float(best_thr)

In [ ]:
threshold = tune_threshold(oof_probs, oof_true)

In [ ]:
from sklearn.metrics import fbeta_score
oof_preds = (oof_probs >threshold).astype(int)

final_f1_score = fbeta_score(y_small, oof_preds, beta=1)
print("\n========== KẾT QUẢ OOF CUỐI CÙNG ==========")
print(f"F1 Score (OOF): {final_f1_score:.4f}")


========== KẾT QUẢ OOF CUỐI CÙNG ==========
F1 Score (OOF): 0.8730


In [ ]:
def cloud_fbeta_df(gt_df, pred_df, beta=2):
    df = gt_df.merge(
        pred_df,
        on=["image_id", "px", "py"],
        suffixes=("_gt", "_pred")
    )

    tp = ((df.label_gt == 1) & (df.label_pred == 1)).sum()
    fp = ((df.label_gt == 0) & (df.label_pred == 1)).sum()
    fn = ((df.label_gt == 1) & (df.label_pred == 0)).sum()

    score = (1 + beta**2) * tp / (
        (1 + beta**2) * tp + beta**2 * fn + fp + 1e-9
    )
    return score


In [ ]:

def safe_div(a, b):
    return a / (b + EPS)

def read_tif_13band(path):
    with rasterio.open(path) as src:
        img = src.read().astype(np.float32)
    return img

def compute_indices(img):
    B = lambda k: img[BAND_MAP[k]]
    return {
        'NDVI': safe_div(B('B08') - B('B04'), B('B08') + B('B04')),
        'NDSI': safe_div(B('B03') - B('B11'), B('B03') + B('B11')),
        'NDWI': safe_div(B('B03') - B('B08'), B('B03') + B('B08')),
        'NDMI': safe_div(B('B08') - B('B11'), B('B08') + B('B11')),
        'B11_B08_Ratio': safe_div(B('B11'), B('B08')),
        'B02_B04_Ratio': safe_div(B('B02'), B('B04')),
        'B8A_B11_Ratio': safe_div(B('B8A'), B('B11')),
        'B01_B11_Ratio': safe_div(B('B01'), B('B11')),
        'B05_B04_Ratio': safe_div(B('B05'), B('B04')),
        'B12_B03_Ratio': safe_div(B('B12'), B('B03')),
    }

def build_feature_matrix(img):
    H, W = img.shape[1], img.shape[2]
    idx_feats = compute_indices(img)

    stack = []
    for name in FEATURE_NAMES:
        if name in BAND_MAP:
            stack.append(img[BAND_MAP[name]])
        else:
            stack.append(idx_feats[name])

    X = np.stack(stack, axis=0).reshape(len(FEATURE_NAMES), -1).T
    return X, H, W



In [ ]:
import os
import numpy as np
import rasterio
from rasterio.windows import Window
import pandas as pd
import joblib
import glob
from sklearn.ensemble import RandomForestClassifier

# ================= CONFIG =================
TIF_DIR = "/content/drive/MyDrive/GEE S2_Cloud_2023_MAX_Data/Test(28 5)"
MODEL_DIR = "models/catboost"  # giữ nguyên tên thư mục
TILE_SIZE = 512
CLOUD_THRESHOLD = threshold
SCALER_PATH = os.path.join(MODEL_DIR, "scaler.pkl")
MODEL_PATTERN = os.path.join(MODEL_DIR, "svm_fold*.joblib")


def load_ensemble_resources():
    """Tải tất cả các mô hình Random Forest đã lưu (giữ tên hàm, đường dẫn)."""

    # Không dùng scaler nữa, bỏ dòng này nếu không cần
    print("No scaler needed for RF.")
    scaler = None

    # Tải các mô hình RF
    model_files = sorted(glob.glob(MODEL_PATTERN))
    if not model_files:
        raise FileNotFoundError(f"No models found in {MODEL_DIR} matching pattern {MODEL_PATTERN}")

    models = []
    for path in model_files:
        model = joblib.load(path)  # load RF/SVM
        models.append(model)

    print(f"Successfully loaded {len(models)} RF models for ensemble prediction.")
    return scaler, models


def predict_folder_to_dataframe(scaler, models):
    """Dự đoán ensemble RF, bỏ qua patch có NaN."""
    records = []
    num_models = len(models)

    tif_files = sorted([f for f in os.listdir(TIF_DIR) if f.endswith(".tif")])
    print(f"Found {len(tif_files)} tif files")

    for tif_name in tif_files:
        tif_path = os.path.join(TIF_DIR, tif_name)
        image_id = os.path.splitext(tif_name)[0]

        print(f"Processing {image_id}")

        with rasterio.open(tif_path) as src:
            H, W = src.height, src.width
            for row in range(0, H, TILE_SIZE):
                for col in range(0, W, TILE_SIZE):
                    h = min(TILE_SIZE, H - row)
                    w = min(TILE_SIZE, W - col)
                    window = Window(col, row, w, h)

                    img = src.read(window=window).astype(np.float32)
                    valid_mask = np.ones((h, w), dtype=bool)

                    X, _, _ = build_feature_matrix(img)

                    # Bỏ patch nếu có NaN
                    if np.isnan(X).any():
                        print(f"Patch at row {row}, col {col} contains NaN, skipping...")
                        continue

                    flat_valid = valid_mask.reshape(-1)
                    X_valid = X[flat_valid]

                    all_probs_valid = np.zeros((len(X_valid), num_models), dtype=np.float32)
                    for i, model in enumerate(models):
                        all_probs_valid[:, i] = model.predict_proba(X_valid)[:, 1]

                    probs_valid_mean = np.mean(all_probs_valid, axis=1)
                    probs = np.zeros(h * w, dtype=np.float32)
                    probs[flat_valid] = probs_valid_mean
                    probs = probs.reshape(h, w)

                    cloud_mask = (probs >= CLOUD_THRESHOLD) & valid_mask

                    ys, xs = np.where(valid_mask)
                    for y, x in zip(ys, xs):
                        records.append([
                            image_id,
                            col + x,
                            row + y,
                            int(cloud_mask[y, x])
                        ])

    submission_df = pd.DataFrame(
        records,
        columns=["image_id", "px", "py", "label"]
    )

    print(f"\nTotal predicted pixels: {len(submission_df)}")
    return submission_df


# ================= RUN =================
try:
    scaler, ensemble_models = load_ensemble_resources()
    submission_df = predict_folder_to_dataframe(scaler, ensemble_models)

    print("\nSubmission DataFrame Head:")
    print(submission_df.head())

except FileNotFoundError as e:
    print(f"ERROR: {e}. Please check MODEL_DIR and file paths.")


No scaler needed for RF.
Successfully loaded 3 RF models for ensemble prediction.
Found 46 tif files
Processing tile_00002_R2048C0
Patch at row 0, col 0 contains NaN, skipping...
Processing tile_00006_R1536C512
Patch at row 0, col 0 contains NaN, skipping...
Processing tile_00007_R2048C512
Processing tile_00008_R2560C512
Processing tile_00009_R3072C512
Processing tile_00010_R3584C512
Patch at row 0, col 0 contains NaN, skipping...
Processing tile_00012_R1024C1024
Patch at row 0, col 0 contains NaN, skipping...
Processing tile_00013_R1536C1024
Processing tile_00014_R2048C1024
Processing tile_00015_R2560C1024
Processing tile_00020_R3072C1536
Processing tile_00021_R3584C1536
Processing tile_00022_R4096C1536
Patch at row 0, col 0 contains NaN, skipping...
Processing tile_00025_R3072C2048
Processing tile_00026_R3584C2048
Patch at row 0, col 0 contains NaN, skipping...
Processing tile_00027_R4096C2048
Patch at row 0, col 0 contains NaN, skipping...
Processing tile_00028_R4608C2048
Patch at r

In [ ]:
TEST_CSV = "/content/drive/MyDrive/GEE S2_Cloud_2023_MAX_Data/Test(28 5)/Output/test.csv"
gt_df = pd.read_csv(TEST_CSV)

print("GT pixels:", len(gt_df))
print("Pred pixels:", len(submission_df))


GT pixels: 12058624
Pred pixels: 7602176


In [ ]:
# ===== COMPUTE SCORE =====
score = cloud_fbeta_df(gt_df, submission_df, beta=2)
print("F2 score:", score)

F2 score: 0.804682230865229


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
X_cnn = X_scaled.reshape(X_scaled.shape[0], 1, X_scaled.shape[1])


In [ ]:
class CNN1D(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.squeeze(-1)
        return self.fc(x)


In [ ]:
def f2_score(pred, target, eps=1e-7):
    pred = (pred > 0.5).float()

    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    fn = ((1 - pred) * target).sum()

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)

    return (5 * precision * recall) / (4 * precision + recall + eps)


In [ ]:
from torch.utils.data import Dataset, DataLoader

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        return x, y



In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
import os

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DIR = "models_cnn1d"
os.makedirs(MODEL_DIR, exist_ok=True)

cv = KFold(n_splits=3, shuffle=True, random_state=42)

oof_probs = np.zeros(len(y))


In [ ]:
for fold, (tr_idx, val_idx) in enumerate(cv.split(X_cnn)):
    print(f"\n========== FOLD {fold} ==========")

    X_tr, X_val = X_cnn[tr_idx], X_cnn[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    train_ds = TabularDataset(X_tr, y_tr)
    val_ds   = TabularDataset(X_val, y_val)

    train_loader = DataLoader(
        train_ds,
        batch_size=8192,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=16384,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    model = CNN1D().to(DEVICE)

    # ===== LOSS xử lý mất cân bằng =====
    pos_weight = torch.tensor(
        (len(y_tr) - y_tr.sum()) / y_tr.sum(),
        dtype=torch.float32
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_f2 = 0.0

    # ================== TRAIN ==================
    for epoch in range(1, 4):
        model.train()
        train_loss = 0.0

        for x, yb in train_loader:
            x = x.to(DEVICE)
            yb = yb.to(DEVICE).unsqueeze(1)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # ================== VALID ==================
        model.eval()
        f2_total = 0.0

        with torch.no_grad():
            for x, yb in val_loader:
                x = x.to(DEVICE)
                yb = yb.to(DEVICE).unsqueeze(1)

                prob = torch.sigmoid(model(x))
                pred = (prob > 0.3).float()
                f2_total += f2_score(pred, yb)

        f2_epoch = f2_total / len(val_loader)

        print(f"Epoch {epoch:02d} | Loss {train_loss:.4f} | Val F2 {f2_epoch:.4f}")

        if f2_epoch > best_f2:
            best_f2 = f2_epoch
            model_path = os.path.join(MODEL_DIR, f"cnn1d_fold{fold}.pth")
            torch.save(model.state_dict(), model_path)
            print(f" Lưu mô hình tốt nhất: {model_path}")

    # ================== OOF (CHUẨN) ==================
    model.eval()

    val_loader_oof = DataLoader(
        val_ds,
        batch_size=16384,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    start = 0
    with torch.no_grad():
        for x, _ in val_loader_oof:
            x = x.to(DEVICE)
            prob = torch.sigmoid(model(x)).cpu().numpy().ravel()
            bs = len(prob)
            oof_probs[val_idx[start:start+bs]] = prob
            start += bs



========== FOLD 0 ==========
Epoch 01 | Loss 186.5237 | Val F2 0.8956
✔ Lưu mô hình tốt nhất: models_cnn1d/cnn1d_fold0.pth
Epoch 02 | Loss 165.8456 | Val F2 0.9134
✔ Lưu mô hình tốt nhất: models_cnn1d/cnn1d_fold0.pth
Epoch 03 | Loss 159.3065 | Val F2 0.8987

========== FOLD 1 ==========
Epoch 01 | Loss 185.0980 | Val F2 0.9110
✔ Lưu mô hình tốt nhất: models_cnn1d/cnn1d_fold1.pth
Epoch 02 | Loss 163.9714 | Val F2 0.9013
Epoch 03 | Loss 158.2648 | Val F2 0.8686

========== FOLD 2 ==========
Epoch 01 | Loss 184.3507 | Val F2 0.8170
✔ Lưu mô hình tốt nhất: models_cnn1d/cnn1d_fold2.pth
Epoch 02 | Loss 165.6271 | Val F2 0.8745
✔ Lưu mô hình tốt nhất: models_cnn1d/cnn1d_fold2.pth
Epoch 03 | Loss 160.0753 | Val F2 0.9067
✔ Lưu mô hình tốt nhất: models_cnn1d/cnn1d_fold2.pth


In [ ]:
from sklearn.metrics import fbeta_score
oof_preds = (oof_probs >0.5).astype(int)

final_f1_score = fbeta_score(y, oof_preds, beta=1)
print("\n========== KẾT QUẢ OOF CUỐI CÙNG ==========")
print(f"F1 Score (OOF): {final_f1_score:.4f}")


========== KẾT QUẢ OOF CUỐI CÙNG ==========
F1 Score (OOF): 0.8426


In [ ]:
TIF_DIR = "/content/drive/MyDrive/GEE S2_Cloud_2023_MAX_Data/Test(28 5)"
MODEL_DIR = "models_cnn1d"
MODEL_PATTERN = os.path.join(MODEL_DIR, "cnn1d_fold*.pth")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TILE_SIZE = 512
CLOUD_THRESHOLD = 0.5  # bạn đã tune F2 rồi


In [ ]:
def load_ensemble_resources():
    model_files = sorted(glob.glob(MODEL_PATTERN))
    if not model_files:
        raise FileNotFoundError("No CNN1D models found")

    models = []
    for path in model_files:
        model = CNN1D()                      # KHÔNG truyền num_features
        state = torch.load(path, map_location=DEVICE)
        model.load_state_dict(state)
        model.to(DEVICE)
        model.eval()                         # inference mode
        models.append(model)

    print(f"Loaded {len(models)} CNN1D models")
    return models


In [ ]:
def predict_folder_to_dataframe(models):
    records = []

    tif_files = sorted([f for f in os.listdir(TIF_DIR) if f.endswith(".tif")])
    print(f"Found {len(tif_files)} tif files")

    for tif_name in tif_files:
        tif_path = os.path.join(TIF_DIR, tif_name)
        image_id = os.path.splitext(tif_name)[0]

        print(f"\nProcessing {image_id}")

        try:
            with rasterio.open(tif_path) as src:
                H, W = src.height, src.width

                for row in range(0, H, TILE_SIZE):
                    for col in range(0, W, TILE_SIZE):

                        try:
                            h = min(TILE_SIZE, H - row)
                            w = min(TILE_SIZE, W - col)
                            window = Window(col, row, w, h)

                            img = src.read(window=window).astype(np.float32)

                            # ===== feature extraction =====
                            X, _, _ = build_feature_matrix(img)
                            if np.isnan(X).any():
                                X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

                            N = X.shape[0]
                            probs_sum = np.zeros(N, dtype=np.float32)

                            # ===== batch CNN1D inference =====
                            for start in range(0, N, BATCH_SIZE):
                                end = min(start + BATCH_SIZE, N)

                                xb = (
                                    torch.from_numpy(X[start:end])
                                    .float()
                                    .unsqueeze(1)  # (B, 1, F)
                                    .to(DEVICE)
                                )

                                with torch.no_grad():
                                    probs_models = []
                                    for model in models:
                                        p = torch.sigmoid(model(xb)).cpu().numpy().ravel()
                                        probs_models.append(p)

                                probs_sum[start:end] = np.mean(probs_models, axis=0)

                            probs = probs_sum.reshape(h, w)
                            cloud_mask = probs >= CLOUD_THRESHOLD

                            ys, xs = np.where(np.ones((h, w), dtype=bool))
                            for y, x in zip(ys, xs):
                                records.append([
                                    image_id,
                                    col + x,
                                    row + y,
                                    int(cloud_mask[y, x])
                                ])

                        except Exception as tile_err:
                            print(f"Skip tile row={row}, col={col} | {tile_err}")
                            continue

        except Exception as file_err:
            print(f"Skip FILE {image_id} | {file_err}")
            continue

    submission_df = pd.DataFrame(
        records,
        columns=["image_id", "px", "py", "label"]
    )

    print(f"\n Total predicted pixels: {len(submission_df)}")
    return submission_df


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

import rasterio
from rasterio.windows import Window
try:
    # suy ra số feature từ 1 patch bất kỳ
    sample_tif = glob.glob(os.path.join(TIF_DIR, "*.tif"))[0]
    with rasterio.open(sample_tif) as src:
        img = src.read(window=Window(0, 0, 32, 32))
        X_sample, _, _ = build_feature_matrix(img)
        num_features = X_sample.shape[1]

    ensemble_models = load_ensemble_resources()
    submission_df = predict_folder_to_dataframe(ensemble_models)

    print(submission_df.head())

except Exception as e:
    print("ERROR:", e)


Loaded 3 CNN1D models
Found 46 tif files

Processing tile_00002_R2048C0

Processing tile_00006_R1536C512

Processing tile_00007_R2048C512

Processing tile_00008_R2560C512

Processing tile_00009_R3072C512

Processing tile_00010_R3584C512

Processing tile_00012_R1024C1024

Processing tile_00013_R1536C1024

Processing tile_00014_R2048C1024

Processing tile_00015_R2560C1024

Processing tile_00020_R3072C1536

Processing tile_00021_R3584C1536

Processing tile_00022_R4096C1536

Processing tile_00025_R3072C2048

Processing tile_00026_R3584C2048

Processing tile_00027_R4096C2048

Processing tile_00028_R4608C2048

Processing tile_00030_R2560C2560

Processing tile_00031_R4608C2560

Processing tile_00036_R3072C3072

Processing tile_00037_R3584C3072

Processing tile_00038_R4096C3072

Processing tile_00039_R4608C3072

Processing tile_00040_R5120C3072

Processing tile_00041_R5632C3072

Processing tile_00043_R2560C3584

Processing tile_00044_R4096C3584

Processing tile_00045_R4608C3584

Processing til

In [ ]:
# ===== COMPUTE SCORE =====
score = cloud_fbeta_df(gt_df, submission_df, beta=2)
print("F2 score:", score)

F2 score: 0.759961297924055
